#Importacion Librerias


In [0]:
from pyspark.sql import functions as F
from datetime import datetime

#Creando Funciones de Apoyo

In [0]:
def get_current_date():
    """
    Get current date
    Returns:
        Current date in yyyy-MM-dd HH:mm:ss.SSS format
    """
    return spark.sql(
        "select date_format(current_timestamp,'yyyy-MM-dd HH:mm:ss.SSS')"
    ).collect()[0][0]



#Invocacion a tabla en Bronze

In [0]:
#obtener los datos de la tabla bronse
df_bronze = spark.read.table("workspace.yelp_bronze.tip")

#Obtenemos la hora de ejecucion
date = get_current_date()
type_proces ='Full'

#Transformacion Datos

In [0]:
#Lectura de los campos requeridos
df_silver = df_bronze.select(
    F.col('user_id').alias('tip_id'),
    F.col('name').alias('nombre'),
    F.col('average_stars').cast('double').alias('promedio_estrellas'),
    F.col('review_count').cast('bigint').alias('n_resenas'),
    F.col('fans').cast('bigint').alias('fans'),
    F.col('elite').alias('elite'),
    F.col('friends').alias('amigos'),
    F.col('yelping_since').cast('timestamp').alias('miembro_desde'),
    F.col('useful').cast('bigint').alias('util'),
    F.col('funny').cast('bigint').alias('divertido'),
    F.col('cool').cast('bigint').alias('cool'),
    F.col('_rescued_data').alias('estado_cuenta'),
    F.lit(date).alias("feccargainfo")
)


#Grabando en tabla Silver

In [0]:

print(f'EL proceso de grabacion es : {type_proces}')
if type_proces =='Full' :
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.yelp_silver.slv_tip")
else :
    df_silver.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable("workspace.yelp_silver.slv_tip")


In [0]:
dbutils.notebook.exit('SUCCESS')

In [0]:
display(df_silver)

In [0]:
display(df_bronze)